# Session 2:  Parallel Processing with MsPASS
## *Gary L. Pavlis*
## *Ian Wang*

## Overview

This notebook is a hands on exercise that students will run during the second session of this class on GeoLab.  We will run and review this notebook while a different, parallel workflow is running on an HPC system at TACC.

The learning objective of this tutorial are:
1.  Understand fundamental concepts of generic schedulers
2.  Understand the concepts of map-reduce
3.  Understand the Futures approach to parallel processing
4.  Understand a typical serial MsPASS workflow
5.  Understand how to convert a serial workflow to a parallel workflow with map-reduce
6.  Understand what a pipeline workflow is and how to use the sliding_window_pipeline function of MsPASS instead of map-reduce.

Note:  MsPASS supports two different "schedulers" for handling parallel processing:  (1) dask, and (2) spark (technically pyspark).  We will only use the default scheduler for MsPASS, dask, for this class for a number of reasons that are not worth relating as it would only confuse the issues.  Furthermore, spark currently lacks a Futures interface needed to implement the sliding_window_pipeline we will utilize in this tutorial.

## Before Running this Notebook
### Launch the database server

As noted in the PrecourseProcessing notebook, an issue we haven't been able to resolve is how to have a private instance of MongoDB launch for each instance of a geolab login.  For that reason you will need to do a shortened version of the commands you used in the PrecourseProcessing notebook.  

Launch a terminal window in GeoLab and enter these commands:
```
cd   #  makes sure you are running this in your home directory
mongod --dbpath ./db --logpath ./logs/mongo_log
```
Note a couple things that can go wrong here:
-  The command will block.  If you want to reuse that terminal window put an & at the end of the mongod launch line.
-  DO NOT exit the terminal window from which you run the mongod command or you will kill the database server.  
-  Each time you reconnect to GeoLab you will need to relaunch the MonogDB server with that  incantation. Be careful you always run the command from the same directory as that way the database files will be written to ~/db and the mongodb log will appear in ~/logs.
-  In all cases it is wise to launch a second Terminal window and verify that worked by typing `ps -A`  You should see a line where the CMD field is "mongod".   If not, contact me by email or slack if you are unable to solve the problem.  The first place to look if you are having a problem is the content of the file ~/logs/mongo_log that will be generated when this command runs.

### Use outside GeoLab
If you are accessing this notebook from github and are not part of the 2025 short course you can still run this tutorial notebook on a desktop system. To do so you will need to do two things:
1.  Install the `mspass-desktop` GUI described in __[MsPASS User's Manual here](https://www.mspass.org/getting_started/mspass_desktop.html)__
2.  Launch MsPASS as described on that page and run this notebook using the "run" button on the GUI. Alternatively, you may run it interactively by pushing the "jupyter" button.

## Initializations
First, the stock incantation that most MsPASS workflows should use for initialization.   

In [ ]:
from mspasspy.db.database import Database
import mspasspy.client as msc
mspass_client = msc.Client()
dask_client = mspass_client.get_scheduler()
db = mspass_client.get_database("ES2026")

Then doing a series of imports needed to support the workflow.  Note common practice is that during development of a workflow in a notebook it is a good idea to keep most imports at the top like this.   As you develop the workflow it is ok to add new imports later, but for a production workflow it is preferable to move them all to the top.  Note, if you are an experienced python programmer you realize there are odd ways that can break your code, but putting all includes together is normally a good idea.  

In [ ]:
from mspasspy.algorithms.window import WindowData,scale
from mspasspy.algorithms.basic import ator,ExtractComponent
from mspasspy.ccore.algorithms.basic import TimeWindow
from mspasspy.ccore.utility import ErrorSeverity
from mspasspy.db.normalize import ObjectIdMatcher,normalize
from mspasspy.algorithms.signals import filter
import matplotlib.pyplot as plt
from mspasspy.graphics import SeismicPlotter
from obspy import UTCDateTime
import time

# TODO

1.  Seismogram Processing section needs to be a separate notebook.   It breaks the logics of this session as it is a side issue.
2.  The serial prototype will need to be altered with the new data but can mostly be retained.  With this new data aligned by a human analyst pick I want to termiante the workflow with a stack operator.
3.  New workflow can illustrate both map and reduce operations.  Should use ensembles, however, instead of atomic operators and implement the stack in a function driven by an input ensemble.
4.  Class will reference a new notebook on how to actually use dasks bag reduce operator called foldby.   

Serial workflow should finish by plotting all the stacks.

Note the serial workflow may need a limit to something like 5 events to be reasonable.   May even be essential to make the map-reduce operator fit in memory. 

Alternatively, could run with a compute by event.  Might be a better lesson as an all in one is a know very bad idea.

Keep the broadband_snr_QC function run but consider an edit step to kill some data.   Might want to compare stack with and without passing through that function.

## Serial Prototype
### Overview
Although one of most important features of MsPASS is it's integrated, generic parallel processing functionality I would assert all users of MsPASS need to internalize two axioms:

1.  Always prototype your workflow as a serial process as the first step.
2.  Only convert that workflow to a parallel workflow if necessary to make the computations feasible.

I will blab more about those two points in the lecture associated with this notebook, but you should view this section of the notebook as an example of item 1.  It also includes simple examples of getting performance data to establish how essential parallel processing is to complete the task.

### Optional Aside
To fully understand the serial workflow we will run below it is helpful, although not essential, to understand these key concepts:

-  The generic concept of a "object" in modern computing and what we mean in MsPASS by a "data object"
-  MsPASS defines the concept of a `Seismogram` object as a container holding the combined channels from a three-component sensor set.
-  A `Seismogram` object has a set of methods that simplify coordinate transformations (e.g. things like rotation to radial, transverse, and vertical from EW, NS, and Z)

The details of what that means are important, but are an aside from the primary topic of today: parallel processing.   Consequently, an asignment for all students in this course is to read through and run the notebook you will find in this directory called "SeismogramAside.ipynb" as part of the homework before the final session of the class.   That tutorial aims to help you better understand those concepts encapsulated in a `Seismogram` object.  For the rest of this session all you really need to know is:
1.  A `Seismogram` is data we will be handling.
2.  You created a bunch of these when you ran the precourse notebooks.
3.  The objects we will be working will be constructed by using readers in the MsPASS database API.
4.  We will be saving the results of calculations to the MsPASS database using different methods of the database API.

### Generic Structure of a Serial Workflow
MsPASS serial workflows nearly always reduce to this pseudocode structure:
```
  1) import all required modules and functions
  2) define all custom python processing functions
  3) create a database query list OR a virtual list with a MongoDB CommandCursor
  4) foreach entry in list:
     5)  read data object from database using entry
     6)  run a series of processing functions on this datum
     7)  save the result to the database using a "data_tag" to identify what it is
```
That hides a lot of details, but I'll use those number tags and refer back to them as we examine the tutorial workflow we will be running momentarily.   

We already did part of 1), but we'll need to add a few.   That is typically how a serial workflow evolves - you know which ones at first you know you will need and then you add additional imports as you realize you need them. 

Now we are at part 2).  There are a lot of details, but you first need to understand what processing we are aiming to do.   i.e. we need to define some functions needed to create generic step 6 above.  To understand what is defined below here is a summary of what the processing steps later will need to do:

- Align the data to P wave pick times made by the Earthscope ANF analysts (i.e. time 0 is the P wave arrival time)
- Make certain the Seismograms are in standard coordinates (E,N,Z)
- Apply Kennett's free surface transformation (see "SeismogramAside" notebook and/or ask your favorite AI for background) to the data
- Extract the "longitudinal component"
- Stack the longitudinal component data (create a "beam" in array processing jargon)

There are then two functions needed that are not just part of MsPASS.  I urge you to just run this next cell to define two functions we will need.  You will see how they are used shortly.   The key point is to realize these two function implement a functionality not wired into MsPASS.   Most workflows will need to create a set of custom functions like this to handle something unique to your problem.   That is actually a very very good thing compared to something like SAC.   The python language is infinitely more flexible than a primitive job control language like SAC's interpreter.   The flexibility of python allows you to implement new approaches that would be impossible with a dinosaur like SAC. 

In [ ]:
import math
from mspasspy.ccore.seismic import SlownessVector
from mspasspy.algorithms.basic import free_surface_transformation
from mspasspy.ccore.utility import ErrorSeverity

def set_PStime(d,
               Ptimekey="Ptime",
               Stimekey="Stime",
               model=None,
               receiver_collection="site",
              ):
    """
    Function to calculate P and S wave arrival time and set times 
    as the header (Metadata) fields defined by Ptimekey and Stimekey.
    Tries to handle some complexities of the travel time calculator 
    returns when one or both P and S aren't calculatable.  That is 
    the norm in or at the edge of the core shadow.  
    Requires source and receiver coordinates stored in the Metadata 
    of d.  Required source data is frozen as "source_lat", 
    "source_lon", "source_depth", and "source_time".   Receiver 
    data is similar but depends upon the value of `receiver_collection`.
    (see below).  If any of that required metadata is missing the 
    return will be killed with a message posted to elog.  
    
    :param d:  input TimeSeries datum.  Assumes datum's Metadata 
      contains stock source and channel attributes.  
    :param Ptimekey:  key used to define the header attribute that 
      will contain the computed P time.  Default "Ptime".
    :param model:  instance of obspy TauPyModel travel time engine. 
      Default is None.   That mode is slow as an new engine will be
      constructed on each call to the function.  Normal use should 
      pass an instance for greater efficiency.  
    :param receiver_collection:   normalizing collection used to 
      load receiver metadata.   In MsPASS normalizing attributes 
      are (by default) loaded with a prefix defining the collection 
      the came from.   The default for this argument is "site" which 
      means the function will attempt to fetch receiver location 
      data with the keys "site_lat" and "site_lon"  
    """
    if d.live:
        if model is None:
            model = TauPyModel(model="iasp91") 
        rlatkey=f"{receiver_collection}_lat"
        rlonkey=f"{receiver_collection}_lon"
        required_keys=["source_lat","source_lon","source_depth","source_time"]
        required_keys.append(rlatkey)
        required_keys.append(rlonkey)
        message = ""
        for key in required_keys:
            if key not in required_keys:
                message += "Missing required key={}\n".format(key)
        if len(message) > 0:
            d.kill()
            d.elog.log_error("setPStime",message,ErrorSeverity.Invalid)
        # extract required source attributes
        srclat=d["source_lat"]
        srclon=d["source_lon"]
        srcz=d["source_depth"]
        srct=d["source_time"] 
        # extract required channel attributes
        stalat=d[rlatkey]
        stalon=d[rlonkey]
        # set up and run travel time calculator
        georesult=gps2dist_azimuth(srclat,srclon,stalat,stalon)
        # obspy's function we just called returns distance in m in element 0 of a tuple
        # their travel time calculator it is degrees so we need this conversion
        dist=kilometers2degrees(georesult[0]/1000.0)
        arrivals=model.get_travel_times(source_depth_in_km=srcz,
                                            distance_in_degree=dist,
                                            phase_list=['P','S'])
        # always post this for as it is not cheap to compute
        # WARNING:  don't use common abbrevation delta - collides with data dt
        d['epicentral_distance']=dist
        # these are CSS3.0 shorthands s - station e - event
        esaz = georesult[1]
        seaz = georesult[2]
        # css3.0 names esax = event to source azimuth; seaz = source to event azimuth
        d['esaz']=esaz
        d['seaz']=seaz
        # get_travel_times returns an empty list if a P time cannot be 
        # calculated.  We trap that condition and kill the output 
        # with an error message
        if len(arrivals)==2:
            Ptime=srct+arrivals[0].time
            rayp = arrivals[0].ray_param
            Stime=srct+arrivals[1].time
            rayp_S = arrivals[1].ray_param
            d.put(Ptimekey,Ptime)
            d.put(Stimekey,Stime)
            # These keys are not passed as arguments but could be - a choice
            # Ray parameter is needed for free surface transformation operator
            # note tau p calculator in obspy returns p=R sin(theta)/V_0
            d.put("rayp_P",rayp)
            d.put("rayp_S",rayp_S)
        elif len(arrivals)==1:
            if arrivals[0].name == 'P':
                Ptime=srct+arrivals[0].time
                rayp = arrivals[0].ray_param
                d.put(Ptimekey,Ptime)
                d.put("rayp_P",rayp)
            else:
                # Not sure we can assume name is S
                if arrivals[0].name == 'S':
                    Stime=srct+arrivals[0].time
                    rayp_S = arrivals[0].ray_param
                    d.put(Stimekey,Stime)
                    d.put("rayp_S",rayp_S)
                else:
                    message = "Unexpected single phase name returned by taup calculator\n"
                    message += "Expected phase name S but got " + arrivals[0].name
                    d.elog.log_error("set_PStime",
                                     message,
                                     ErrorSeverity.Invalid)
                    d.kill()
                
    # Note python indents mean if an ensemble is marked dead this function just silenetly returns 
    # what it received doing nothing - correct mspass model
    return d

def apply_FST(d,rayp_key="rayp_P",seaz_key='seaz',vp0=6.0,vs0=3.5):
    """
    Apply free surface transformation operator of Kennett (1983) to an input `Seismogram` 
    object.   Assumes ray parameter and azimuth data are stored as Metadata in the 
    input datum.  If the ray parameter or azimuth key are not defined an error 
    message will be posted and the datum will be killed before returning.  
    :param d:  datum to process
    :type d:  Seismogram
    :param rayp_key:   key to use to extract ray parameter to use to compute the 
    free surface transformation operator.  Note function assumes the ray parameter is
    spherical coordinate form:  R sin(theta)/V.   Default is "rayp_P".
    :param seaz_key:   key to use to extract station to event azimuth. Default is "seaz".
    :param vp0:  surface P wave velocity (km/s) to use for free surface transformation 
    :param vs0:  surface S wave velocity (km/s) to use for free surface transformation.
    """
    if d.is_defined(rayp_key) and d.is_defined(seaz_key):
        rayp = d[rayp_key]
        seaz = d[seaz_key]
        # Some basic seismology here.  rayp is the spherical earth ray parameter
        # R sin(theta)/v.  Free surface transformation needs apparent velocity 
        # at Earth's surface which is sin(theta)/v when R=Re.   Hence the following
        # simple convertion to get apparent slowness at surface  sin(theta)/v
        Re=6378.1
        umag = rayp/Re   # magnitude of slowness vector
        # seaz is back azimuth - slowness vector points in direction of propagation
        # with is 180 degrees away from back azimuth
        az = seaz + 180.0
        # component slowness vector components in local coordinates
        ux = umag * math.sin(az)
        uy = umag * math.cos(az)
        # FST implementation requires this special class called a Slowness Vector
        u = SlownessVector(ux,uy)
        d = free_surface_transformation(d,uvec=u,vp0=vp0,vs0=vs0)
    else:
        d.kill()
        message = "one of required attributes rayp_P or seaz were not defined for this datum"
        d.elog.log_error("apply_FST",message,ErrorSeverity.Invalid)
        
    return d

We can now define our complete serial workflow.  Here it is.   Start it running and scroll to the next box of this notebook.  While this runs I'll highlight topics there and we will discuss some of the details.   You will have questions and this will be a good time to ask them while this runs.

In [1]:
srcid_list=db.wf_Seismogram.distinct("source_id")
site_matcher = ObjectIdMatcher(db,
    collection="site",
    attributes_to_load=["_id","lat","lon","elev"]
source_matcher = ObjectIdMatcher(db,
    collection="source",
    attributes_to_load=["lat","lon","depth","time","_id"],
  )
print("Number of ensembles to process=",len(srcid_list))
nlive_saved = 0
n_dead = 0
t0 = time.time()
for srcid in srcid_list:
    # the data_tag section of this query is not needed if the wf_Seismogram collection is pristine
    # used it here to make this section more robust if the notebook is rerun after altering wf_Seismogram later in the course
    query = {"source_id" : srcid, "data_tag" : {"$exists" : False} }
    cursor = db.wf_Seismogram.find(query)
    # adds source_id to ensembles metadata container
    ens_metadata = {"source_id" : srcid}
    ens = db.read_data(cursor,collection="wf_Seismogram",ensemble_metadata=ens_metadata)
    # post the source_id to ensemble's metadata area
    if ens.live:
    ens = normalize(ens,site_matcher)
    ens = normalize(ens,source_matcher)
    ens = rotate_to_standard(ens)
    # run the set_PStime function we defined above on each "member" of the ensemble
    # note it also sets the slowness metadata required by applyFST.   That is run in the same loop since that 
    # function, like set_PStime, works only on atomic data
    for i in range(len(ens.member)):
        # change the default keys or we will overwrite Ptime set from ANF picks - this is a class metadata problem to beware
        ens.member[i] = set_PStime(ens.member[i],Ptimekey="Ptime_iasp91",Stimekey="Stime_iasp91")
        ens.member[i] = applyFST(ens.member[i])
    # Shift the data so time 0 is the pick time made by ANF NOT the time computed from the model
    ens = ator(ens,"Ptime")
    # this extracts the longitudinal component created by transformation in applyFST
    ens = ExtractComponent(ens,2)   
    # stack the L components - normalize by the number of live data
    N=0
    stack = None
    for d in ens.member:
        if d.live:
            if N==0:
                stack = Seismogram(d)
                N = 1
            else:
                stack += d
                N += 1
    if stack is None:
        print("No live data for source_id=",srcid)
        print("Not result for this ensemble")
        continue
    stack /= float(N)
    
    # add the new Metaata attribute "fold" - a reflection seismology jargon term
    stack["fold"] = N
    if stack.live:
        nlive_saved += 1   # assumes the next line succeeds
    sdret = db.save_data(stack,collection="wf_TimeSeries",data_tag="simple_stack")  # collection argument is not essential here, but needed for clarity in this tutorial
t = time.time()
print("Number of stacked data saved=",nlive_saved)
print("Run time=",t-t0)

SyntaxError: unterminated string literal (detected at line 49) (2285991647.py, line 49)

In [ ]:
# verify this agrees with print statement immediately above - useful sanity check
# mismatch may happen if you rerun one of the notebook code boxes
n=db.wf_Seismogram.count_documents({"data_tag" : "simple_stack"})
print("Current number of stacks in database = ",n)

A few key points here to link back to my pseudocode structure above:

-  The structure of the main loop for this processing (step 3 of the pseudocode) is a common one for handling data grouped into "ensemble" containers.   Since our objective here is producing stacks of data from the usarray from common sources, the grouping is by "source_id".  I used the "distinct" method of MongoDB to only select source_ids with waveform data.  The processing loop is then an iteration over that list of source_ids.
-  The reading stage has to convert the source_id to a valid MongoDB query.  The line `cursor = db.wf_Seismogram.find(query)` applies that query.
-  The MsPASS database reader run as `db.read_data(cursor,collection="wf_Seismogram")` automatically recognizes that when it receives a CommandCursor it should load an ensemble of data.   In this workflow I use the symbol "ens" as a reference to that result.
-  I use the inline `normalize` function to load source and receiver metadata. This workflow cannot run without that as the get_PStime function we defined above requires the attibutes that function loads.
-  See the comments for what the processing steps are doing

## Questions?

### QC sanity check
We should look at what we have here.  We could make tables of Metadata, but tables aren't something we human's can digest easily.   That is why the generic field of "computer visualization" is important.  Here, we'll use the mspass `SeismicPlotter` to visualize our stacked data as it simplifies plotting seismic data objects.  

Plotting all the stacked data would be overwhelming and not particularly useful out of the context of the data that generated the stacks.  Data from the smaller events may not have a very clear signal without filtering even after stacking.   For this tutorial a simple solution to this is to select only the data that have the highest "fold" == number of data summed for that average.   The biggest events "light up" the whole array and will have a pick on almost every station.  Smaller events will be seen only on the quietest stations.  We'll visualize that by creating a histogram of the values of the "fold" attribute we set above and which are now stored with the stacks.   

In [ ]:
query={"data_tag" : "simple_stack"}
fold_list=list()
cursor = db.wf_TimeSeries.find(query)
for doc in cursor:
    # best to be safe rather than sorry with this test
    if "fold" in doc:
        fold_list.append(doc["fold"])
plt.hist(fold_list)
plt.show()

We can use that plot to select only data with the highest fold

In [ ]:
plotter=SeismicPlotter(normalize=True)   # normalize essential here as the stacks are variable magnitude inputs
minimum_fold = 100    # see histogram above for guidance on size
query = {"fold" : {"$gt" : minimum_fold}
cursor = db.wf_Seismogram.find(query)
ens = db.read_data(cursor)
plotter.plot(ens)

## Parallel Workflow
### Some General Concepts
For this tutorial we will be using the parallel scheduler called [dask](https://www.dask.org/).  Dask is a generic, open-source, parallel processing system.   It has many features outside of the scope of this class.  For this class there are two central features of dask you will need to understand to make sense out of the rest of this exercise:
1.   Dasks abstracts the idea of a "cluster" of machines, which we will have discussed in the class prior to starting this exercise.  The top-level model of parallelism in dask is what is called the foreman (master) - worker (slave) model of parallel processing.   That means basically workers (slaves) take orders from a foreman (master) that control which worker does what tasks.  In dask's documentation the "foreman" is always called the "scheduler", but the concept is the same.  The scheduler assigns tasks to workers and manages the flow of data to and from the workers.   How much work can be accomplished is limited by the number of workers (normally the same as the number of cores assigned to the job) available to do the computations.  For more on these generic concepts [the following](https://www.geeksforgeeks.org/parallel-algorithm-models-in-parallel-computing/) is a pretty good introduction to the fundamentals.  Dask has a suite of abstractions of how to set that up in what they call [dask distributed](https://distributed.dask.org/en/stable/).   Today we will be using a version appropriate to the environment we are running called [LocalCluster]().  You are probably unaware that an instance of the dask scheduler was created for you in the first code box of this notebook when we created the thing stored with the symbol *mspass_client*.  We will be exploiting that momentarily.
2.   Once we have a local cluster create we can submit work to it.  Dask is a generic system for parallel processing.  One of the concept it can exploit is the generic concept called [map-reduce (MapReduce)](https://en.wikipedia.org/wiki/MapReduce).  MsPASS processing algorithms are all based on the map-reduce model.   The jargon hides a simple perspective on how to implement a parallel algorithm with map-reduce we will discuss shortly.  There are huge numbers of additional sources on this generic concept you can find with a simple web search.  For this class you may also want to read the pages with the heading "System Tuning" in [MsPASS User Manual](https://www.mspass.org).

### Creating or connecting to a cluster
#### For users running this tutorial with docker
If you are running this notebook outside of GeoLab with docker the following section will not work.  The docker container launches dask in a different way that make the approach below fail.   The reasons, as the saying goes, are "deep in the weeds" but it should be viewed as just a different way to build a virtual cluster.   The docker container version simply uses a different approach that is not appropriate for GeoLab.  If you are running with docker see the [this section](http://www.mspass.org/getting_started/run_mspass_with_docker.html) of the MsPASS User's Manual for guidance.  If you are running this way skip to the next subsection.

#### For students in Earthscope course
As noted above we will need to interact with an (abstact) instance of a cluster to which we can submit work.   For this tutorial we will be using dask's [LocalCluster](https://docs.dask.org/en/stable/deploying-python.html).  As noted above it was actually created for you (under the hood as the saying goes) in the first code box when we instantiated an instance of the MsPASS [Client](http://www.mspass.org/python_api/mspasspy.client.html) object we assigned the symbol `mspass_client`.   To access the scheduler we use a "method" of the client similar to the `get_database` method we used earlier called, appropriately, `get_schduler`.   This one line code box does that:

In [ ]:
dask_client = mspass_client.get_scheduler()

It is instructive to view an overview of its content with this command:

In [ ]:
dask_client.cluster

On GeoLab you should be able to click on that link to connect to the so called "dask diagnostic" (status) display.   Students should verify that works as we will be watching that real-time display when we run our parallel job below.  For the present it should open a static display with a series of graphic displays we will discuss later collectively.  

![title](img/DaskStatus0.png)

A common need is to change the number of worker processes assigned to the job.  The virtual machine we are using for this class has 4 cores allocated to each user.   Look at the "Status" page on the dask diagnostics window you just opened.  Notice that the graphic titled "Bytes stored per worker" has four blue bars.  Convert the (currently disabled) cell to a "Code" box and run the following to change from 4 to 10 workers and notice what happens:

What changes in the display?

You could run with that configuration BUT for this particular application that is heavily compute-bound the processing would probably take significantly longer as the number of process would exceed the number of cores available.  The different tasks would all be fighting for the same resource.  For that reason we want to restore the original 4 worker configuration with the following:

In [ ]:
dask_client.cluster.scale(4)

In [ ]:
import numpy as np
import dask.bag as dbg
rng = np.random.default_rng(seed=42)   # 42 is the secret to life, the universe, and everything (Hitchhiker's guide to the Galaxy)
seeds = rng.uniform(low=0.0, high=1000000, size=100000)
mybag = dbg.from_sequence(seeds,npartitions=20)
print("The type of mybag is ",type(mybag))

Define some functions that will alter the bag.

In [ ]:
def make_random_vector(float_seed,N,loc=0.0,scale=1.0,)->np.ndarray:
    """
    Create a vector of length N of Gaussian distributed noise with sigma=scale and mean=loc.   
    arg0 (float_seed) is a floating point number to use as a seed for the random number 
    generator.  In this tutorial it is itself generated from a random number generator
    using a particular seed (42).  As the name suggests it is expected to be a float but 
    it is truncated to an int that is used for the randoms eed.
    """
    rng = np.random.default_rng(seed = int(float_seed))  
    d = rng.normal(loc=loc,scale=scale,size=N)
    return d
def sumsq(d):
    """
    Return sum of squares of input numpy vector d.   Numpy doesn't have an explicit function 
    for this but what we use here is vectorized and very fast.
    """
    return np.sum(d*d)

In [ ]:
Nsamp = 1000   # number of samples per vector
mybag = mybag.map(make_random_vector,Nsamp)
mybag = mybag.map(sumsq)
print("The type of the output of the sequence of map operator is ",type(mybag))

Section discussing the idea of a lazy/delayed computation.   

In [ ]:
result = mybag.compute()

In [ ]:
import matplotlib.pyplot as plt
print("Size of result vector=",len(result))
plt.hist(result)

## Parallel Processing Run
Here is a parallel version of the same processing sequence as the prototype serial code box above. The lecture will have discussed how to translate a serial job to a parallel job.  Before running this box compare the processing lines here to the serial prototype above to be sure you understand that key ida.  

When we tell you to go push the run box on the following and immediately jump to the dask diagnostics page.  We will be exploring the content of that page in real time while the job below runs:  

TODO:   need to explain why these auxiliary functions are needed

In [ ]:
def query_generator(sid):
    return {"source_id" : sid}
def set_PStime_ensemble(ens):
    for i in range(len(ens.member)):
        ens.member[i] = set_PStime(ens.member[i],Ptimekey="Ptime_iasp91",Stimekey="Stime_iasp91")
    return ens
def applyFST_ensemble(ens):
    for i in range(len(ens.member)):
        ens.member[i] = applyFST(ens.member[i])
    return ens
def stack_ensemble(ens):
    N=0
    stack = None
    for d in ens.member:
        if d.live:
            if N==0:
                stack = TimeSeries(d)
                N = 1
            else:
                stack += d
                N += 1

    # with serial workflow we could just post a message here
    # in a parallel workflow we have to return something 
    # this illustrates why mspass uses an ErrorLogger container and kills to handle this 
    # kind of data problem
    if stack is None:
        stack = TimeSeries()
        stack.kill()   # not strictly necessary, but better to be clear this datum is marked dead
        # this conditional makes this function more robust - source_id might not be defined
        if "source_id" in ens:
            stack["source_id"] = ens["source_id"]
        message("No live data found to stack.   No stack computed for this source_id")
        stack.elog.log_error("stack_ensemble",message,ErrorSeverity.Invalid)
    else:
        stack /= float(N)
    stack["fold"] = N   # note this will be 0 for killed data
    return stack
    

In [ ]:
import dask.bag as dbg
import os

# repeated for clarity
srcid_list=db.wf_Seismogram.distinct("source_id")
site_matcher = ObjectIdMatcher(db,
    collection="site",
    attributes_to_load=["_id","lat","lon","elev"]
source_matcher = ObjectIdMatcher(db,
    collection="source",
    attributes_to_load=["lat","lon","depth","time","_id"],
  )
print("Number of ensembles to process=",len(srcid_list))

t0=time.time()
dataset = dbg.from_sequence(srcid_list)
dataset = dataset.map(query_generator)
dataset = dataset.map(lambda q : db.wf_Seismogram.find(d))
dataset = dataset.map(db.read_data,collection='wf_Seismogram')
dataset = dataset.map(normalize,site_matcher)
dataset = dataset.map(normalize,source_matcher)
dataset = dataset.map(rotate_to_standard)
dataset = dataset.map(set_PStime_ensemble)
dataset = dataset.map(applyFST_ensemble)
dataset = dataset.map(ator,"Ptime")
dataset = dataset.map(ExtractComponent,2)
dataset = dataset.map(stack_enemble)
dataset = dataset.map(db.save_data,collection="wf_TimeSeries",data_tag="simple_stack_parallel")
dataset=dataset.compute()
t = time.time()
print("Time to process = ",t-t0)

### Diagnosing Performance of Dask
First, let's take a closer look at the Dask Diagnostic page, accessible via the URL of the cluster mentioned earlier. This page displays real-time performance data for the tasks currently running on Dask.

![title](img/DaskStatus1.png)

#### Progress Panel
The Progress panel provides a visual representation of task completion, with bars indicating the status of partitions being processed. For instance, if you have configured the from_sequence method with npartitions=50, each map operation will show 51 partitions (including an additional partition for handling any remainder).

Here's what the progress might look like after some time:

![title](img/DaskStatus2.png)

As tasks progress, you might notice that only three steps are depicted in the panel. This is due to Dask's optimization behavior where it might merge several map functions into fewer ones to simplify the processing graph.

#### Workers Tab
Under the Workers tab, you can monitor the status of each of the four workers engaged in processing tasks:

![title](img/DaskWorkerStatus3.png)

#### Profiling Tab
The Profile tab provides a real-time profile of the tasks being executed, showing the execution stack aggregated across all workers and threads. The horizontal axis represents the percentage of time spent on specific functions, while the vertical axis shows the execution call stack:

![title](img/DaskProfile4.png)
![title](img/DaskProfile5.png)

Note: The profiling in Dask is performed through sampling. Therefore, functions that execute very briefly may not appear in the profile. However, this sampled data is still useful for pinpointing slower sections of the workflow.

#### Graph and Group Tabs
The Graph tab visualizes the processing graph, which in our case is straightforward since it primarily involves a series of map operations. The Group tab displays the same graph but groups individual nodes from the same step together:

![title](img/DaskGraph6.png)
![title](img/DaskGroup7.png)

The following prints provide some useful supplementary information discussed below:

In [ ]:
print('The type of the data returned by the bag compute method is ',type(dataset))
print('This list is of size=',len(dataset))
print('Content of the first list component:  ',dataset[0])

Some key points to glean from the code above:
1.   There is, mostly, a one-to-one correspondence between lines of the serial processing prototype and the parallel version above.   That is, each function call in the serial job is replaced by a call the "map" operator.    The exception is that for the serial job we intentionally avoided saving anything.  Saving the data, for this example, was a bit more complex because we chose to also save the result as files.   As we discussed in session  2 with files with different names.  We needed to specified `storage_mode=file` as before but we also inserted the small function that sets the dir and dfile attributes to organize the files with different names - in this case common source gathers with dfile names defined the the source_id.
2.   The serial workflow is driven by a loop over a `CommandCursor`.  That is replaced in this code by the dask "bag" method `from_sequence`.   In this context what that line does is effectively create a list of documents retrieved by MongoDB that is stored in the "bag"  we reference with the symbol `dataset`.   You should think of a dask bag as a python list container that may not fit in memory.   In this example, on creation with the `from_sequence` line, `dataset` would fit in disk.   The second line that calls the `read_data` method, however, changes that converts the bag to a huge list that is the data set that is to be processed.
3.   The "npartitions" argument passed to `from_sequence` determines the chunk size of the bag.   The "Graph" item in dask diagnostics shows you that dask treats this job as 50 pipelines that run the processing sequence on blocks of data of the approximate size N/50 where N is the number of Seismogram objects to process.  
4.   The last line of the script, which calls the "compute" method of bag, initiates what is commonly called a "lazy computation".  The newer dask documentation talks about this in terms of what it calls a [Futures](https://docs.dask.org/en/latest/futures.html) object.   In that jargon `compute` submits the actual work to the cluster that turn the `Futures` object into the output you request.
5.   The extra box at the end shows that `compute` returns a python list not a "bag".   Naive use of compute can cause memory faults as discussed in the dask documentation [here](https://distributed.dask.org/en/stable/memory.html).  Since v2 of MsPASS the default for all seismic data writers is to return only a summary of what was saved to reduce the incidence of memory errors from inappropriate use of the bag compute method.   Most seismic processing workflows terminate in a save so this model should be followed for most MsPASS workflows. 

### Parallel IO variant
Dask (and pyspark) are generic systems to parallelize a variety of different types of algorithms.   The type of algorithm we just ran is an example of a classic one called a pipeline as we reviewed in the lecture done in parallel with this notebook.   Because dask is generic there are multiple ways to accomplish exactly the same result.   The next two code boxes illustrate two variants.

The first variant is a necessary evil for very large data sets.   The reason is a subtle problem we learned early on about using MongoDB in this context.   That is, if a job like the above can be subject to a [cursor timeout](https://www.mongodb.com/docs/manual/reference/method/cursor.maxTimeMS/) problem.  For that reason we wouldn't normally recommend a construct like above, but we used it because it makes clearer the one-to-one relationship of steps in the serial workflow and parallel equivalent.  In addition to the "cursor timeout" problem we also found that MongoDB reads and writes can cause io bottlenecks.   That issue is more accute on more lightweight calculations than the one we will do here. The balance of I/O and compute time in processing large data sets is a fundamental issue with subtleties far outside the scope of this class. Some fundamental material on computer IO are found in the [I/O in MsPASS](http://www.mspass.org/user_manual/io.html) section of the User's Manual and many online sources.   Parallel I/O and how the motivation for this variant are found in the related section titled [Paralle IO in MsPASS](http://www.mspass.org/user_manual/parallel_io.html) that may be helpful in understand the motivation for creating this alternative method for handling IO in MsPASS.  The code below uses the special MsPASS reader called [read_distributed_data](http://www.mspass.org/python_api/mspasspy.io.html#mspasspy.io.distributed.read_distributed_data) to replace the read block and [write_distributed_data]() to replace the write section.  Consider reading those sections at your leisure.  For now, examine this variant and note the differences just stated.  The only other difference here is the line `dataset = dataset.compute()` is not used in this script.   The reason is that `write_distributed_data` actually contains lines that initiate the "lazy computation" and return a summary of the results similar to that we got above.   Note this code block is disabled initially to allow this notebook to be run from start to end without creating the duplicate copies that would be present if this and the next box were actually run.  If you want to examine their behavior or run timing tests on the menu above change "Raw" to "Code".  

### Combining process functions
The second variant uses a different trick that is more useful for clarity and code reuse than performance.   This approach may also be useful for pipelines subject to memory bloat problems that we have found dask is prone to for many workflow (see [this section](http://www.mspass.org/user_manual/memory_management.html) of the User Manual).  

First, what this variant does is something dask actually does automatically for you but in a more opaque way.   That is, if you examine the "Graph" menu item in dask diagnostics you will see three vertical bars that display the processing progress.  The left bar is the reader and the right bar is the writer.   The processing functions are all lumped together as a single "task" in the middle bar.   The code below illustrates more-or-less what dask does automatically to create that processing geometry;  it combines all the pipeline functions to the equivalent of the single `do_all` function run in this variant.  

One difference this variant shows is a key reason using this variant may be advised in come cases.  Note we don't recycle the data symbol, *d*, but have the functions output to a different variable.  When that symbol is no longer active we use the `del` command of python to force releasing the memory attached to that symbol.  What that does is a form of manual memory management.  Without that approach large data objects can accumulate quickly in memory during processing.   Python uses a "garbage collection" algorithm to manage memory, but dask mysteriously decides itself when it should force garbage collection.   We have found that with very large data objects like the ensembles we worked with earlier that it is very easy to create a job that will fail with a memory fault.   The approach shown here is one way to avoid that if you are experiencing a memory fault problem.  

Note like variant 1 this box is turned off by default.  You can change it to "Code" and run it to compare it to the earlier run box.  

### Tuning
Results may vary a bit but if you actually ran the three different variants above you would find that for this processing the first job and variant 2 are pretty much indistinguishable.  Our tests indicate both are within 10% of ideal scaling for 4 workers.   Students should do that calculation for your result using partial processing as a serial job as a benchmark to estimate the parallel time.  Why is the serial job not a perfect baseline and how could you improve it?

The above also illustrate a general issue in tuning an algorithm.   If you run the parallel I/O (variant 1) version, you will find it is actually much slower by roughly a factor of 2.  Dask gives us a hint why that is so.   If you run it you get a message warning of the problem of "Sending large graph of size 18.95 MiB".  That, however, is only a hint.   The lesson to take from that is that tuning a particular workflow requires digging into the "Profile" tab while the job is running or saving a static graphic of the profiler output at from a test run.  Comparing the profiler output for the different approaches above would likely give us hints about why the parallel I/O algorithm is so slow.  For this example, the timing numbers alone tell us we are unlikely to do much better with further tuning. Whether or not the effort to do that tuning is justified is dependent upon the scale of the problem you are facing. e.g. if you are only going to do something once or twice and you find the computation is feasible with a serial workflow, you probably don't need to mess with the complexities of tuning a parallel workflow.  If, however, you estimate your job will require weeks to months of processing with a serial job, or you need to run a workflow on dozens of data sets each of which requires days, the effort to transfer the workflow to a large cluster is justified.  For guidance on tuning dask see the sections of the MsPASS User's manual on [parallel processing](http://www.mspass.org/user_manual/parallel_processing.html) and consult numerous web sources on the topic.  

Finally, it is important to realize that results will also vary with hardware.  A particularly important one is that if you don't handle some details the parallel job above will actually take longer to run than the serial job on a typical desktop/laptop system.  Why, is largely an issue of a mismatch in default configurations for docker and dask with what we just ran.  The biggest problem, at present, seems to be that dask, by default, is using what is called a "thread pool" shared by the processing tasks that fight each other.  The conflict comes from an obscure python concept called the "Global Interpreter Lock" (GIL) that in the default configuration allows only one thread to execute simultaneously.   As a result, the above parallel algorithms can easily be signficantly slower than the serial job because the GIL allows only one thread to execute simultaneously making it effectively a serial job with the additional dask overhead.   The additional overhead is the scheduler and time spent moving data between tasks.   The solution on a desktop is to use the elaborate run line noted near the top of this notebook for running with docker that contains this incantation:
```
-e MSPASS_WORKER_ARG="--nworkers 4 --nthreads 1"
```
That particular problem is less likely to surface when running on an HPC cluster for a number of reasons, but the real lesson is this:  optimizing parallel performance often requires digging deeper and doing some tuning to scheduler (dask or spark) configurations.   The GIL problem is just another not so trivial details that can bite you in scaling a workflow to run on a large cluster. 

## new section on sliding window algorithm

- Lecture material will need to cover reasons
- Monitoring workflow section above may need to include a performance report to monitor memory use.   Definitely will need to discuss in real time as the job runs.
- This section should replace map calls by a process function and use an accumulator for stacking. 